# HỎI ĐÁP TRÊN ẢNH TÀI LIỆU

Notebook baseline độc lập; toàn bộ mã nguồn nằm trong các cell bên dưới.


## 1. Cấu hình đường dẫn


In [ ]:
import os

# Phải đặt trước khi torch khởi tạo CUDA.
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

from pathlib import Path
import time
import zipfile

import torch

# Cố định kết quả giữa các lần chạy: cùng seed -> cùng file nộp.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

ROOT = Path.cwd().resolve().parent
DATA = ROOT / 'data'
TRAIN_DIR = DATA / 'training_set'
RUNS = ROOT / 'outputs'
RUNS.mkdir(parents=True, exist_ok=True)
SPLIT = 'public_test'  # Đổi thành 'private_test' trong phase private.

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[cấu hình] ROOT={ROOT} | SPLIT={SPLIT}')
print('[cấu hình] thiết bị: ' + (f'{DEVICE} ({torch.cuda.get_device_name(0)})'
      if DEVICE.type == 'cuda' else str(DEVICE)))


## 2. Định nghĩa mô hình


In [ ]:
# Các mô hình khởi tạo ngẫu nhiên cho tác vụ; không kèm trọng số huấn luyện sẵn.
import torch
from torch import nn


ROUTER_CLASSES = (
    "lookup", "count", "compare", "sum", "argmax", "argmin",
    "cross_page_sum", "visual_bold_lookup",
)


class QuestionRouterCNN(nn.Module):
    """Train-only character CNN that selects the reasoning program."""

    def __init__(self, vocab: int = 256, width: int = 64, classes: int = len(ROUTER_CLASSES)):
        super().__init__()
        self.embedding = nn.Embedding(vocab, width, padding_idx=0)
        self.convolutions = nn.ModuleList(
            nn.Conv1d(width, width, kernel_size, padding=kernel_size // 2)
            for kernel_size in (3, 5, 7)
        )
        self.output = nn.Sequential(
            nn.Linear(width * 3, width * 2), nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(width * 2, classes),
        )

    def forward(self, character_ids: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(character_ids).transpose(1, 2)
        pooled = [torch.relu(layer(embedded)).amax(-1) for layer in self.convolutions]
        return self.output(torch.cat(pooled, dim=-1))


class CharEncoder(nn.Module):
    def __init__(self, vocab: int, width: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab, width, padding_idx=0)
        self.rnn = nn.GRU(width, width // 2, batch_first=True, bidirectional=True)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        sequence, _ = self.rnn(self.embedding(ids))
        return sequence.mean(dim=-2)


class CropCNN(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.projection = nn.Linear(128, width)

    def forward(self, crops: torch.Tensor) -> torch.Tensor:
        shape = crops.shape
        values = self.net(crops.flatten(0, 1)).flatten(1)
        return self.projection(values).view(shape[0], shape[1], -1)


class MultimodalDocQANet(nn.Module):
    """Fuse question, OCR characters, image crops and normalized bbox."""

    def __init__(self, char_vocab: int = 256, operations: int = 8, width: int = 192):
        super().__init__()
        self.question = CharEncoder(char_vocab, width)
        self.ocr = CharEncoder(char_vocab, width)
        self.visual = CropCNN(width)
        self.layout = nn.Sequential(nn.Linear(5, width), nn.ReLU(), nn.Linear(width, width))
        layer = nn.TransformerEncoderLayer(width, 6, width * 4, batch_first=True, norm_first=True)
        self.fusion = nn.TransformerEncoder(layer, 3)
        self.question_projection = nn.Linear(width, width)
        self.evidence_head = nn.Linear(width, 1)
        self.answer_cell_head = nn.Linear(width, 1)
        self.operation_head = nn.Linear(width, operations)
        self.bold_head = nn.Linear(width, 2)

    def forward(
        self,
        question_ids: torch.Tensor,
        ocr_char_ids: torch.Tensor,
        crops: torch.Tensor,
        bbox_page: torch.Tensor,
        padding_mask: torch.Tensor | None = None,
    ) -> dict[str, torch.Tensor]:
        batch, blocks, chars = ocr_char_ids.shape
        question = self.question(question_ids)
        ocr = self.ocr(ocr_char_ids.view(batch * blocks, chars)).view(batch, blocks, -1)
        tokens = ocr + self.visual(crops) + self.layout(bbox_page)
        tokens = tokens + self.question_projection(question).unsqueeze(1)
        fused = self.fusion(tokens, src_key_padding_mask=padding_mask)
        pooled = fused.masked_fill(padding_mask.unsqueeze(-1), 0).sum(1) / (~padding_mask).sum(1, keepdim=True).clamp_min(1) if padding_mask is not None else fused.mean(1)
        return {
            "evidence_logits": self.evidence_head(fused).squeeze(-1),
            "answer_cell_logits": self.answer_cell_head(fused).squeeze(-1),
            "operation_logits": self.operation_head(pooled),
            "bold_logits": self.bold_head(fused),
        }


class OCRCorrectionCRNN(nn.Module):
    """Optional image-to-character correction branch for noisy OCR crops."""

    def __init__(self, classes: int = 256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 48, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(48, 96, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(96, 192, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, None)),
        )
        self.rnn = nn.GRU(192, 160, num_layers=2, bidirectional=True, batch_first=True)
        self.output = nn.Linear(320, classes)

    def forward(self, crops: torch.Tensor) -> torch.Tensor:
        features = self.cnn(crops).squeeze(2).transpose(1, 2)
        sequence, _ = self.rnn(features)
        return self.output(sequence).log_softmax(-1)


def parameter_report() -> dict[str, dict[str, float]]:
    models = {
        "QuestionRouterCNN": QuestionRouterCNN(),
        "MultimodalDocQANet": MultimodalDocQANet(),
        "OCRCorrectionCRNN": OCRCorrectionCRNN(),
    }
    return {
        name: {
            "parameters": sum(value.numel() for value in model.parameters()),
            "fp32_mib": sum(value.numel() for value in model.parameters()) * 4 / 2**20,
            "fp16_mib": sum(value.numel() for value in model.parameters()) * 2 / 2**20,
        }
        for name, model in models.items()
    }



## 3. Tiện ích đọc dữ liệu


In [ ]:
# Tiện ích đọc OCR và bố cục trang. Baseline chỉ cài ba phép suy luận đơn giản:
# lookup, count và tổng hai dòng. Các kiểu còn lại là phần việc của đội thi.
from dataclasses import dataclass
from itertools import product
import json
from pathlib import Path
import re
from typing import Iterable


@dataclass
class DocumentLayout:
    document_id: str
    blocks: list[dict]


def read_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def load_split(split_dir: Path) -> tuple[list[dict], dict[str, DocumentLayout]]:
    manifests = read_jsonl(split_dir / "manifest.jsonl")
    questions = read_jsonl(split_dir / "questions.jsonl")
    layouts: dict[str, DocumentLayout] = {}
    for manifest in manifests:
        payload = json.loads((split_dir / manifest["ocr_path"]).read_text(encoding="utf-8"))
        layouts[manifest["id"]] = DocumentLayout(
            document_id=manifest["id"],
            blocks=[block for page in payload["pages"] for block in page["blocks"]],
        )
    return questions, layouts


def center(block: dict) -> tuple[float, float]:
    x1, y1, x2, y2 = block["bbox"]
    return (x1 + x2) / 2, (y1 + y2) / 2


def block_width(block: dict) -> float:
    return block["bbox"][2] - block["bbox"][0]


def parse_number(text: str) -> float | None:
    value = text.strip().replace(" ", "").rstrip("%")
    if not re.fullmatch(r"[+-]?[0-9][0-9.,]*", value):
        return None
    if "," in value:
        value = value.replace(".", "").replace(",", ".")
    else:
        value = value.replace(".", "")
    try:
        return float(value)
    except ValueError:
        return None


def format_number(value: float) -> str:
    if abs(value - round(value)) <= 1e-9:
        return str(int(round(value)))
    return f"{value:.2f}".rstrip("0").rstrip(".").replace(".", ",")


def public_evidence(blocks: Iterable[dict]) -> list[dict]:
    seen: set[str] = set()
    result: list[dict] = []
    for block in blocks:
        if block["block_id"] in seen:
            continue
        seen.add(block["block_id"])
        result.append({"page": block["page"], "bbox": block["bbox"]})
    return result


def page_blocks(layout: DocumentLayout, page: int) -> list[dict]:
    return [block for block in layout.blocks if int(block["page"]) == page]


def table_blocks(layout: DocumentLayout, page: int, table_index: int) -> list[dict]:
    blocks = page_blocks(layout, page)
    titles = sorted(
        [
            block for block in blocks
            if block_width(block) >= 0.65 and "BẢNG" in str(block["text"]).upper()
        ],
        key=lambda item: item["bbox"][1],
    )
    if not titles:
        return blocks if table_index == 1 else []
    if not 1 <= table_index <= len(titles):
        return []
    start = titles[table_index - 1]["bbox"][1] - 1e-6
    end = titles[table_index]["bbox"][1] - 1e-6 if table_index < len(titles) else 1.0
    return [block for block in blocks if start <= center(block)[1] < end]


def group_rows(blocks: list[dict]) -> list[list[dict]]:
    grouped: dict[float, list[dict]] = {}
    for block in blocks:
        grouped.setdefault(round(float(block["bbox"][1]), 6), []).append(block)
    return [sorted(values, key=lambda item: center(item)[0]) for _, values in sorted(grouped.items())]


def header_block(blocks: list[dict], text: str) -> dict | None:
    candidates = [block for block in blocks if str(block["text"]) == text]
    return min(candidates, key=lambda item: item["bbox"][1]) if candidates else None


def cell_under(row: list[dict], header: dict) -> dict | None:
    header_x, _ = center(header)
    candidates = [
        block for block in row
        if block["bbox"][0] - 1e-6 <= header_x <= block["bbox"][2] + 1e-6
    ]
    if candidates:
        return min(candidates, key=lambda item: abs(center(item)[0] - header_x))
    return min(row, key=lambda item: abs(center(item)[0] - header_x)) if row else None


def data_rows(blocks: list[dict], headers: list[dict]) -> list[list[dict]]:
    if not headers:
        return []
    boundary = max(header["bbox"][3] for header in headers)
    return [
        row for row in group_rows(blocks)
        if min(block["bbox"][1] for block in row) >= boundary - 1e-6
        and any(block["bbox"][3] - block["bbox"][1] < 0.06 for block in row)
    ]


def parse_condition_pairs(text: str) -> list[tuple[str, str]]:
    matches = list(re.finditer(r"“([^”]+)”", text))
    pairs: list[tuple[str, str]] = []
    previous_end = 0
    for match in matches:
        header = text[previous_end : match.start()].strip()
        header = re.sub(r"^(?:và|với)\s+", "", header, flags=re.IGNORECASE)
        header = re.sub(r"^(?:dòng\s+có|đối\s+với)\s+", "", header, flags=re.IGNORECASE)
        header = header.strip(" ,:.;")
        if not header:
            return []
        pairs.append((header, match.group(1)))
        previous_end = match.end()
    return pairs


def matching_rows(blocks: list[dict], pairs: list[tuple[str, str]]) -> list[tuple[list[dict], list[dict]]]:
    if not pairs:
        return []
    headers: list[dict] = []
    for header_text, _ in pairs:
        header = header_block(blocks, header_text)
        if header is None:
            return []
        headers.append(header)
    matches: list[tuple[list[dict], list[dict]]] = []
    for row in data_rows(blocks, headers):
        cells = [cell_under(row, header) for header in headers]
        if all(cell is not None and str(cell["text"]) == value for cell, (_, value) in zip(cells, pairs, strict=True)):
            matches.append((row, [cell for cell in cells if cell is not None]))
    return matches


def solve_lookup(question: str, layout: DocumentLayout) -> tuple[str, list[dict]] | None:
    patterns = [
        r"Trong bảng \d+ ở trang \d+, (.+?) của dòng có (.+?) là gì\?",
        r"Hãy cho biết (.+?) tại bảng \d+ ở trang \d+ đối với (.+?)\.",
        r"Tại bảng \d+ ở trang \d+, dòng (.+?) ghi (.+?) bằng bao nhiêu\?",
    ]
    target_header = condition_text = None
    for index, pattern in enumerate(patterns):
        match = re.fullmatch(pattern, question)
        if match:
            if index == 2:
                condition_text, target_header = match.group(1), match.group(2)
            else:
                target_header, condition_text = match.group(1), match.group(2)
            break
    if not target_header or not condition_text:
        return None
    page_match = re.search(r"trang\s+(\d+)", question)
    page = int(page_match.group(1)) if page_match else 1
    quoted = re.findall(r"“([^”]+)”", condition_text)
    if not quoted:
        return None
    options: list[list[dict]] = []
    for value in quoted:
        candidates = [
            block for block in layout.blocks
            if int(block["page"]) == page and str(block["text"]) == value
        ]
        if not candidates:
            return None
        options.append(candidates)
    solutions: list[tuple[float, dict, list[dict]]] = []
    for combination in product(*options):
        condition_cells = list(combination)
        y_values = [center(block)[1] for block in condition_cells]
        if max(y_values) - min(y_values) > 0.035:
            continue
        row_y = sum(y_values) / len(y_values)
        headers = [
            block for block in layout.blocks
            if int(block["page"]) == page
            and str(block["text"]) == target_header
            and center(block)[1] < row_y
        ]
        if not headers:
            continue
        header = max(headers, key=lambda item: center(item)[1])
        header_x, header_y = center(header)
        tolerance = max(0.018, max(block["bbox"][3] - block["bbox"][1] for block in condition_cells))
        answer_candidates = [
            block for block in layout.blocks
            if int(block["page"]) == page
            and abs(center(block)[1] - row_y) <= tolerance
            and block not in condition_cells
            and block != header
        ]
        if not answer_candidates:
            continue
        answer_cell = min(
            answer_candidates,
            key=lambda item: abs(center(item)[0] - header_x) + 10.0 * abs(center(item)[1] - row_y),
        )
        score = abs(center(answer_cell)[0] - header_x) + 10.0 * abs(center(answer_cell)[1] - row_y) + 0.02 * (row_y - header_y)
        solutions.append((score, answer_cell, condition_cells))
    if not solutions:
        return None
    _, answer_cell, condition_cells = min(solutions, key=lambda item: item[0])
    return str(answer_cell["text"]), public_evidence(condition_cells + [answer_cell])


def solve_count(question: str, layout: DocumentLayout) -> tuple[str, list[dict]] | None:
    match = re.fullmatch(r"Có bao nhiêu dòng trong bảng (\d+) ở trang (\d+) có (.+?) là “([^”]+)”\?", question)
    if not match:
        return None
    table_index, page = int(match.group(1)), int(match.group(2))
    blocks = table_blocks(layout, page, table_index)
    header = header_block(blocks, match.group(3))
    if header is None:
        return None
    matches: list[dict] = []
    for row in data_rows(blocks, [header]):
        cell = cell_under(row, header)
        if cell is not None and str(cell["text"]) == match.group(4):
            matches.append(cell)
    return (str(len(matches)), public_evidence(matches)) if matches else None


def split_two_conditions(blocks: list[dict], text: str):
    pairs = parse_condition_pairs(text)
    for split_at in range(1, len(pairs)):
        first = matching_rows(blocks, pairs[:split_at])
        second = matching_rows(blocks, pairs[split_at:])
        if len(first) == len(second) == 1 and first[0][0] != second[0][0]:
            return first[0], second[0]
    return None


def solve_sum(question: str, layout: DocumentLayout) -> tuple[str, list[dict]] | None:
    match = re.fullmatch(r"Trong bảng (\d+) ở trang (\d+), tổng (.+?) của hai dòng có (.+?) là bao nhiêu\?", question)
    if not match:
        return None
    blocks = table_blocks(layout, int(match.group(2)), int(match.group(1)))
    rows = split_two_conditions(blocks, match.group(4))
    header = header_block(blocks, match.group(3))
    if rows is None or header is None:
        return None
    first_cell = cell_under(rows[0][0], header)
    second_cell = cell_under(rows[1][0], header)
    first = parse_number(first_cell["text"]) if first_cell else None
    second = parse_number(second_cell["text"]) if second_cell else None
    if first is None or second is None:
        return None
    evidence = rows[0][1] + rows[1][1] + [first_cell, second_cell]
    return format_number(first + second), public_evidence(block for block in evidence if block is not None)


SOLVERS = {"lookup": solve_lookup, "count": solve_count, "sum": solve_sum}


def solve(question: str, layout: DocumentLayout, intent: str):
    solver = SOLVERS.get(intent)
    return solver(question, layout) if solver else None


## 4. Bộ định tuyến câu hỏi


In [ ]:
# Bộ định tuyến câu hỏi: một CNN ký tự đoán câu hỏi thuộc kiểu suy luận nào.

import json
import random

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 20260813
MAX_CHARACTERS = 256
CLASS_TO_ID = {name: index for index, name in enumerate(ROUTER_CLASSES)}
STARTER_INTENTS = {'lookup', 'count', 'sum'}


def encode(text: str) -> torch.Tensor:
    values = [min(255, ord(character) % 256) for character in text[:MAX_CHARACTERS]]
    return torch.tensor(values + [0] * (MAX_CHARACTERS - len(values)), dtype=torch.long)


class RouterDataset(Dataset):
    def __init__(self, train_dir: Path):
        questions = {
            item["question_id"]: item
            for item in (
                json.loads(line)
                for line in (train_dir / "questions.jsonl").open(encoding="utf-8")
                if line.strip()
            )
        }
        labels = [
            json.loads(line)
            for line in (train_dir / "labels.jsonl").open(encoding="utf-8")
            if line.strip()
        ]
        self.rows = [
            (encode(questions[label["question_id"]]["question"]), CLASS_TO_ID[label["reasoning_type"]])
            for label in labels
        ]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


class NeuralRouter:
    def __init__(self, checkpoint: Path):
        payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
        self.model = QuestionRouterCNN()
        self.model.load_state_dict(payload["state_dict"])
        self.model.eval()
        self.max_characters = int(payload["max_characters"])
        self.classes = tuple(payload["classes"])

    def predict(self, question: str) -> str:
        values = [min(255, ord(character) % 256) for character in question[: self.max_characters]]
        values.extend([0] * (self.max_characters - len(values)))
        with torch.inference_mode():
            logits = self.model(torch.tensor([values], dtype=torch.long))
        return self.classes[int(logits.argmax(-1).item())]


## 5. Huấn luyện


In [ ]:
ROUTER_PATH = RUNS / 'question_router.pt'
EPOCHS = 2
BATCH_SIZE = 128
LEARNING_RATE = 2e-3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

dataset = RouterDataset(TRAIN_DIR)
generator = torch.Generator().manual_seed(SEED)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
print(f'[huấn luyện] {len(dataset)} câu hỏi | epochs={EPOCHS} | batch={BATCH_SIZE} | seed={SEED}')

router_model = QuestionRouterCNN().to(DEVICE)
optimizer = torch.optim.AdamW(router_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
print(f'[huấn luyện] tham số mô hình: {sum(p.numel() for p in router_model.parameters()):,}')

for epoch in range(1, EPOCHS + 1):
    router_model.train()
    correct = seen = 0
    started = time.time()
    for characters, labels in loader:
        characters, labels = characters.to(DEVICE), labels.to(DEVICE)
        logits = router_model(characters)
        loss = criterion(logits, labels)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        correct += int((logits.argmax(-1) == labels).sum())
        seen += len(labels)
    print(f'[huấn luyện] epoch {epoch}/{EPOCHS} | accuracy={correct / max(seen, 1):.4f} '
          f'| {time.time() - started:.1f}s')

ROUTER_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    'state_dict': router_model.state_dict(),
    'classes': list(ROUTER_CLASSES),
    'max_characters': MAX_CHARACTERS,
    'seed': SEED,
}, ROUTER_PATH)
print(f'[huấn luyện] đã lưu {ROUTER_PATH}')


## 6. Sinh dự đoán


In [ ]:
PREDICTIONS_PATH = RUNS / 'predictions.jsonl'

router = NeuralRouter(ROUTER_PATH)
questions, layouts = load_split(DATA / SPLIT)
print(f'[dự đoán] {len(questions)} câu hỏi trên {len(layouts)} tài liệu ({SPLIT})')

solved = 0
started = time.time()

with PREDICTIONS_PATH.open('w', encoding='utf-8') as handle:
    for item in questions:
        intent = router.predict(item['question'])
        # Baseline chỉ trả lời được ba kiểu suy luận đơn giản nhất.
        result = solve(item['question'], layouts[item['document_id']], intent) \
            if intent in STARTER_INTENTS else None
        answer, evidence = result if result else ('không xác định', [])
        solved += int(result is not None)
        handle.write(json.dumps({
            'question_id': item['question_id'],
            'answer': answer,
            'evidence': evidence,
        }, ensure_ascii=False, sort_keys=True) + '\n')

print(f'[dự đoán] trả lời được {solved}/{len(questions)} câu | {time.time() - started:.1f}s')


## 7. Đóng gói file nộp


In [ ]:
SUBMISSION_PATH = RUNS / f'submission_{SPLIT}.zip'

with zipfile.ZipFile(SUBMISSION_PATH, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(PREDICTIONS_PATH, 'predictions.jsonl')
print(f'[nộp bài] đã tạo {SUBMISSION_PATH}')
